# Challenge 05 - Digit Recognition (MLP)

Recognition of the digit (0-9) pronounced in each audio track with MLPs (`Dense` and `Dropout` layers only) trained with TensorFlow / Keras. The input comes from the MFCCs without temporal collapse. Two kinds of networks are combined:

- **crop networks**: the speech part of the track is cropped using the energy (c0) and resampled to a fixed number of time steps. Each track is cropped in several ways and all the crops are used as training patterns;
- **window networks**: a fixed-size window slides over the speech part of the track and the network classifies every window.

The final classifier averages the output probabilities of the two groups. Each test track is classified independently of the others. The predictions are saved in `Es7Predictions.txt`.

## Setup

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import warnings
warnings.filterwarnings('ignore', message='urllib3 v2 only supports OpenSSL')

import numpy as np
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

# Configuration
db_path = '../../DBs/AudioDigits_ML26_27'
train_file = 'audio_ml2627_train.pkl'
test_file = 'audio_ml2627_test_no_labels.pkl'
result_path = '../Es7Predictions.txt'

# Speech detection
smooth_width = 5        # moving average (frames) applied to c0 before the speech detection
min_speech_frames = 8   # shorter detections are discarded (the whole track is used)

# Crop networks: (mode, threshold) of the crops of each track
n_steps = 16            # time steps of the resampled crop
CROPS = [('span', 0.55), ('span', 0.65), ('span', 0.75), ('span', 0.85),
         ('peak', 0.40), ('peak', 0.55), ('peak', 0.70)]
SINGLE_CROP = [('span', 0.75)]   # reference: one crop per track
n_crop_models = 5
crop_epochs = 100
crop_batch = 32

# Window networks: one group of networks per threshold
half_window = 20        # frames on each side of the central frame
window_step = 2         # one frame every window_step inside the window
WINDOW_FRACS = [0.65, 0.75, 0.85]
n_window_models = 2
window_epochs = 30
window_batch = 128

# MLP
hidden_units = (256, 128)
dropout_in = 0.2
dropout_hidden = 0.4
l2_reg = 1e-3
learning_rate = 1e-3

seed = 1234

## Helper Functions

Dataset loading, feature extraction, model definition, training, prediction and export.

Everything is computed from the track alone:

- **Speech detection**: c0 is smoothed and compared with the relative threshold `min(c0) + frac * (max(c0) - min(c0))`, which adapts to the noise level of each recording. In `span` mode the speech goes from the first to the last frame above the threshold; in `peak` mode it is the contiguous region around the energy peak.
- **Normalization**: the mean of c1-c19 over the speech is subtracted (cepstral mean normalization) and c0 is replaced by c0 minus its maximum.
- **Crop features**: the speech segment is linearly interpolated to `n_steps` frames and flattened; the log of its length is appended. The best threshold changes with the noise, so each training track is cropped with all the `CROPS` and every crop is a training pattern with the label of the track. For a test track the probabilities of its own crops are averaged.
- **Window features**: for every speech frame, the frames in a window of `2 * half_window + 1` frames around it (one every `window_step`) are flattened. Every window is a training pattern with the label of the track. The probability of a track is the softmax of the mean log-probability of its windows.

The scaler is fitted on the training patterns only.

In [2]:
def load_dataset(db_path, filename):
    return np.load(os.path.join(db_path, filename), allow_pickle=True)


def speech_bounds(m, frac, mode):
    pad = smooth_width // 2
    c0 = np.convolve(np.pad(m[:, 0], pad, mode='edge'), np.ones(smooth_width) / smooth_width, mode='valid')
    thr = c0.min() + frac * (c0.max() - c0.min())
    if mode == 'span':
        idx = np.where(c0 >= thr)[0]
        lo, hi = idx[0], idx[-1]
    else:
        lo = hi = c0.argmax()
        while lo > 0 and c0[lo - 1] >= thr:
            lo -= 1
        while hi < len(c0) - 1 and c0[hi + 1] >= thr:
            hi += 1
    if hi - lo + 1 < min_speech_frames:
        return 0, len(m) - 1
    return lo, hi


def normalize(m, lo, hi):
    energy = m[:, :1] - m[lo:hi + 1, 0].max()
    coef = m[:, 1:] - m[lo:hi + 1, 1:].mean(axis=0)
    return np.hstack([energy, coef])


def resample(m, n):
    old, new = np.linspace(0, 1, len(m)), np.linspace(0, 1, n)
    return np.stack([np.interp(new, old, m[:, j]) for j in range(m.shape[1])], axis=1)


def crop_features(m, mode, frac):
    m = np.asarray(m, dtype=float)
    lo, hi = speech_bounds(m, frac, mode)
    seg = normalize(m, lo, hi)[lo:hi + 1]
    return np.append(resample(seg, n_steps).ravel(), np.log(len(seg)))


def window_features(m, frac):
    m = np.asarray(m, dtype=float)
    lo, hi = speech_bounds(m, frac, 'span')
    x = np.pad(normalize(m, lo, hi), ((half_window, half_window), (0, 0)), mode='edge')
    return np.array([x[t:t + 2 * half_window + 1:window_step].ravel() for t in range(lo, hi + 1)])


def build_model(n_features):
    reg = tf.keras.regularizers.l2(l2_reg)
    layers = [tf.keras.layers.Input(shape=(n_features,)), tf.keras.layers.Dropout(dropout_in)]
    for units in hidden_units:
        layers.append(tf.keras.layers.Dense(units, activation='relu', kernel_initializer='he_normal', kernel_regularizer=reg))
        layers.append(tf.keras.layers.Dropout(dropout_hidden))
    layers.append(tf.keras.layers.Dense(10, activation='softmax'))
    model = tf.keras.Sequential(layers)
    model.compile(loss='sparse_categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(learning_rate), metrics=['accuracy'])
    return model


def fit_predict(x_tr, y_tr, x_te_list, n, epochs, batch):
    scaler = StandardScaler().fit(x_tr)
    x_tr = scaler.transform(x_tr)
    x_te_list = [scaler.transform(x) for x in x_te_list]
    proba = [0] * len(x_te_list)
    for i in range(n):
        tf.keras.backend.clear_session()
        tf.keras.utils.set_random_seed(seed + i)
        model = build_model(x_tr.shape[1])
        model.fit(x_tr, y_tr, epochs=epochs, batch_size=batch, shuffle=True, verbose=0)
        for j, x in enumerate(x_te_list):
            proba[j] = proba[j] + model.predict(x, verbose=0, batch_size=1024) / n
    return proba


def crop_proba(m_tr, y_tr, m_te, crops):
    x_tr = np.vstack([[crop_features(m, mode, frac) for m in m_tr] for mode, frac in crops])
    x_te = [np.array([crop_features(m, mode, frac) for m in m_te]) for mode, frac in crops]
    proba = fit_predict(x_tr, np.tile(y_tr, len(crops)), x_te, n_crop_models, crop_epochs, crop_batch)
    return sum(proba) / len(crops)


def window_proba(m_tr, y_tr, m_te):
    total = 0
    for frac in WINDOW_FRACS:
        w_tr = [window_features(m, frac) for m in m_tr]
        w_te = [window_features(m, frac) for m in m_te]
        labels = np.repeat(y_tr, [len(w) for w in w_tr])
        track = np.repeat(np.arange(len(w_te)), [len(w) for w in w_te])
        proba = fit_predict(np.vstack(w_tr), labels, [np.vstack(w_te)], n_window_models, window_epochs, window_batch)[0]
        logp = np.zeros((len(w_te), 10))
        np.add.at(logp, track, np.log(proba + 1e-6))
        logp = logp / np.bincount(track)[:, None]
        e = np.exp(logp - logp.max(axis=1, keepdims=True))
        total = total + e / e.sum(axis=1, keepdims=True)
    return total / len(WINDOW_FRACS)


def save_predictions(predictions, path):
    np.savetxt(path, np.asarray(predictions, dtype=int), fmt='%i')
    print(f'Predictions successfully saved to {path}')

## Data Loading

The training set contains sessions 1 and 4 (620 tracks), the test set sessions 2 and 3 (620 tracks, without labels). The label is the pronounced digit (`digits`). The speaker (`person_ids`) and the compact `features` are not used: only the `mfccs`.

In [3]:
train_data = load_dataset(db_path, train_file)
test_data = load_dataset(db_path, test_file)

train_m = list(train_data['mfccs'])
test_m = list(test_data['mfccs'])
train_y = np.array(train_data['digits'], dtype=np.int32)
train_sess = np.array(train_data['session_ids'])

print('Training tracks: %d, test tracks: %d' % (len(train_y), len(test_m)))
print('Tracks per digit:', np.bincount(train_y))
print('Frames per track: %d-%d' % (min(len(m) for m in train_m), max(len(m) for m in train_m)))

Training tracks: 620, test tracks: 620
Tracks per digit: [62 62 62 62 62 62 62 62 62 62]
Frames per track: 79-745


## Model: Ensemble of MLPs

All the networks have the same structure: `Dropout(0.2)` on the input, two `Dense` layers (256 and 128 units, ReLU, He initialization, L2 regularization) each followed by `Dropout(0.4)`, and a softmax output with 10 units. They are trained with Adam for a fixed number of epochs (there is no early stopping because the validation data are other sessions, not a random hold-out).

- Crop networks: 5 networks with different seeds, 321 inputs (16 time steps x 20 coefficients + log length).
- Window networks: 2 networks for each of the 3 thresholds, 420 inputs (21 frames x 20 coefficients).

The final prediction is the argmax of the mean of the two probabilities.

In [4]:
n_crop = crop_features(train_m[0], *CROPS[0]).shape[0]
n_window = window_features(train_m[0], WINDOW_FRACS[0]).shape[1]
print('Crop network inputs: %d, window network inputs: %d' % (n_crop, n_window))
build_model(n_crop).summary()

Crop network inputs: 321, window network inputs: 420


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dropout (Dropout)               │ (None, 321)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        82,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 116,618 (455.54 KB)

 Trainable params: 116,618 (455.54 KB)

 Non-trainable params: 0 (0.00 B)

## Model Evaluation

The split is done by session to simulate new recording conditions: train on one session and validate on the other, in both directions. The crop networks with a single crop per track are reported as a reference, to show the effect of training on several crops. The choices (crops, window size, thresholds) were made on this same validation, so the final accuracy is slightly optimistic.

In [5]:
cv = {}
for a, c in [(1, 4), (4, 1)]:
    m_tr = [m for m, s in zip(train_m, train_sess) if s == a]
    m_va = [m for m, s in zip(train_m, train_sess) if s == c]
    y_tr, y_va = train_y[train_sess == a], train_y[train_sess == c]
    single = crop_proba(m_tr, y_tr, m_va, SINGLE_CROP)
    crop = crop_proba(m_tr, y_tr, m_va, CROPS)
    window = window_proba(m_tr, y_tr, m_va)
    cv[(a, c)] = (y_va, {'crop networks, single crop': single, 'crop networks': crop,
                         'window networks': window, 'crop + window networks': (crop + window) / 2})
    print('Train session %d -> validation session %d done' % (a, c))

Train session 1 -> validation session 4 done
Train session 4 -> validation session 1 done


In [6]:
names = list(cv[(1, 4)][1])
for name in names:
    accs = [accuracy_score(y, proba[name].argmax(axis=1)) for y, proba in cv.values()]
    print('%-28s 1->4: %.4f   4->1: %.4f   mean: %.4f' % (name, accs[0], accs[1], np.mean(accs)))

crop networks, single crop   1->4: 0.7677   4->1: 0.8097   mean: 0.7887
crop networks                1->4: 0.8774   4->1: 0.9516   mean: 0.9145
window networks              1->4: 0.8903   4->1: 0.9097   mean: 0.9000
crop + window networks       1->4: 0.9194   4->1: 0.9548   mean: 0.9371


## Prediction and Export

The crop and window networks are retrained on all training tracks (sessions 1 and 4), and the same preprocessing (features and scaler fitted on the training set) is applied to the test set. Each test track is classified independently of the others. The predictions are saved in `Es7Predictions.txt`, one digit per line.

In [7]:
proba = (crop_proba(train_m, train_y, test_m, CROPS) + window_proba(train_m, train_y, test_m)) / 2
predictions = proba.argmax(axis=1)

assert len(predictions) == 620
print('Predicted tracks per digit:', np.bincount(predictions, minlength=10))
save_predictions(predictions, result_path)

Predicted tracks per digit: [60 71 60 61 50 62 57 72 70 57]
Predictions successfully saved to ../Es7Predictions.txt
